In [ ]:
%pip install -e .

In [ ]:
import importlib.util
import logging
from pathlib import Path

logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)


def check_editable_installation(package_name: str) -> bool:
    """
    프로젝트가 Editable(-e) 모드로 설치되었는지 검증합니다.
    
    Args:
        package_name: pyproject.toml의 project.name 값
        
    Returns:
        설치가 정상이면 True, 아니면 False
    """
    try:
        spec = importlib.util.find_spec(package_name)
        if spec is None:
            raise ModuleNotFoundError(
                f"'{package_name}' 패키지를 찾을 수 없습니다."
            )

        package_origin = spec.origin
        logger.info(f"패키지 로드 성공 — 연결 경로: {package_origin}")

        # Editable 모드 여부: 소스 원본 경로를 직접 참조하는지 확인
        # Notebook 환경에서는 __file__ 이 정의되지 않으므로 별도 처리
        try:
            current_dir = Path(__file__).resolve().parent
            if str(current_dir) in str(package_origin):
                logger.info("✓ Editable(-e) 모드로 소스 원본과 실시간 연결되어 있습니다.")
        except NameError:
            # Jupyter Notebook 환경에서는 __file__ 미정의 — 경로 검증 생략
            logger.info("✓ Notebook 환경에서는 경로 검증을 건너뜁니다.")

        return True

    except ModuleNotFoundError as e:
        logger.error(f"패키지 미설치: {e}")
        print("\n[조치] 프로젝트 루트(pyproject.toml 위치)에서 아래 명령을 실행하세요:")
        print("  pip install -e .")
        return False

    except Exception as e:
        logger.error(f"예상치 못한 오류: {e}")
        return False

check_editable_installation("rag_core")
check_editable_installation("api")